# Advanced Coding Model Fine-Tuning

Fine-tune Qwen2.5-Coder-7B on advanced coding, architecture, and testing patterns.

## Topics Covered:
- Design Patterns (Repository, Circuit Breaker, CQRS, Strategy)
- System Architecture (Microservices, Multi-tenant SaaS, Event Sourcing)
- Advanced Testing (Unit, Integration, E2E with TestContainers, Playwright)

## 1. Install Dependencies

In [ ]:
# Install Unsloth and dependencies
!pip install "unsloth[cu121-torch240] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install "transformers==4.44.2" --quiet
!pip install datasets accelerate bitsandbytes --quiet

print("✅ Dependencies installed")

## 2. Upload Training Data

Upload your `advanced_combined_raw.jsonl` file:
1. Click the folder icon on the left
2. Upload the file
3. Or use the code cell below to download from a URL

In [ ]:
# Option 1: Upload manually via the file browser
# Option 2: Download from a URL (replace with your URL)
# !wget -O training_data.jsonl "YOUR_URL_HERE"

# Check if file exists
import os
if os.path.exists("advanced_combined_raw.jsonl"):
    print("✅ Training data found")
else:
    print("⚠️ Please upload advanced_combined_raw.jsonl")

## 3. Load and Prepare Data

In [ ]:
import json
from datasets import Dataset

# Load training data
data = []
with open("advanced_combined_raw.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

print(f"📊 Loaded {len(data)} training examples")

# Format for training
def format_example(example):
    instruction = example.get("instruction", "")
    output = example.get("output", "")
    
    # Alpaca format
    text = f"""### Instruction:
{instruction}

### Response:
{output}
"""
    
    return {"text": text}

formatted_data = [format_example(ex) for ex in data]
dataset = Dataset.from_list(formatted_data)

print(f"✅ Dataset prepared: {len(dataset)} examples")
print("\nSample:")
print(dataset[0]["text"][:500] + "...")

## 4. Load Base Model

In [ ]:
from unsloth import FastLanguageModel
import torch

# Model configuration
max_seq_length = 4096
dtype = None  # Auto-detect
load_in_4bit = True  # Use 4-bit quantization for Colab

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-7B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("✅ Base model loaded")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 5. Setup LoRA Fine-tuning

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # LoRA rank
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("✅ LoRA adapters added")

## 6. Configure Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,  # Adjust based on your data size
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

print("✅ Trainer configured")
print(f"Training steps: {trainer.args.max_steps}")
print(f"Batch size: {trainer.args.per_device_train_batch_size}")
print(f"Learning rate: {trainer.args.learning_rate}")

## 7. Train the Model

In [ ]:
# Start training
print("🚀 Starting training...")
trainer_stats = trainer.train()

print(f"\n✅ Training complete!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"Peak memory: {max_memory:.2f} GB" if (max_memory := torch.cuda.max_memory_allocated() / 2**30) else "")

## 8. Test the Model

In [ ]:
# Inference
FastLanguageModel.for_inference(model)

test_prompts = [
    "Create a Python class implementing the Repository Pattern with Unit of Work for database abstraction.",
    "Design a scalable microservices architecture for an e-commerce platform handling 100k+ orders/day.",
    "Write comprehensive unit tests with pytest for a UserService including mocking and fixtures.",
]

for prompt in test_prompts:
    print("\n" + "="*50)
    print(f"Prompt: {prompt[:60]}...")
    print("="*50)
    
    inputs = tokenizer([
        f"### Instruction:\n{prompt}\n\n### Response:\n"
    ], return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens = 512,
        use_cache = True,
    )
    
    response = tokenizer.batch_decode(outputs)[0]
    print(response[len(inputs[0][0]):][:500] + "...")

## 9. Save Model (GGUF Format for Ollama)

In [ ]:
# Save as GGUF for Ollama
model.save_pretrained_gguf(
    "advanced_coder_model",
    tokenizer,
    quantization_method = ["q4_k_m", "q5_k_m", "q8_0"],
)

print("✅ Model saved in GGUF format")
print("\nFiles created:")
!ls -lh advanced_coder_model/*.gguf

## 10. Download Model

Download the GGUF files to use with Ollama locally.

In [ ]:
from google.colab import files

# Download all GGUF files
for f in ["advanced_coder_model-q4_k_m.gguf", "advanced_coder_model-q8_0.gguf"]:
    if os.path.exists(f):
        print(f"Downloading {f}...")
        files.download(f)

print("\n✅ Download complete!")

## Next Steps

1. **Use with Ollama:**
```bash
# Create Modelfile
cat > Modelfile << EOF
FROM ./advanced_coder_model-q4_k_m.gguf
PARAMETER temperature 0.2
PARAMETER num_predict 4000
SYSTEM You are an expert software engineer specializing in advanced design patterns, system architecture, and testing.
EOF

# Create model
ollama create advanced-coder -f Modelfile

# Use it
ollama run advanced-coder
```

2. **Update your lab config:**
```bash
lab model add advanced-coder ollama/advanced-coder
lab agent update code --model advanced-coder
```